This software is Copyright © 2024 The Regents of the University of California. All Rights Reserved. Permission to copy, modify, and distribute this software and its documentation for educational, research and non-profit purposes, without fee, and without a written agreement is hereby granted, provided that the above copyright notice, this paragraph and the following three paragraphs appear in all copies. Permission to make commercial use of this software may be obtained by contacting:

Office of Innovation and Commercialization

9500 Gilman Drive, Mail Code 0910

University of California

La Jolla, CA 92093-0910

(858) 534-5815

invent@ucsd.edu

This software program and documentation are copyrighted by The Regents of the University of California. The software program and documentation are supplied “as is”, without any accompanying services from The Regents. The Regents does not warrant that the operation of the program will be uninterrupted or error-free. The end-user understands that the program was developed for research purposes and is advised not to rely exclusively on the program for any reason.

IN NO EVENT SHALL THE UNIVERSITY OF CALIFORNIA BE LIABLE TO ANY PARTY FOR DIRECT, INDIRECT, SPECIAL, INCIDENTAL, OR CONSEQUENTIAL DAMAGES, INCLUDING LOST PROFITS, ARISING OUT OF THE USE OF THIS SOFTWARE AND ITS DOCUMENTATION, EVEN IF THE UNIVERSITY OF CALIFORNIA HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGE. THE UNIVERSITY OF CALIFORNIA SPECIFICALLY DISCLAIMS ANY WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE. THE SOFTWARE PROVIDED HEREUNDER IS ON AN “AS IS” BASIS, AND THE UNIVERSITY OF CALIFORNIA HAS NO OBLIGATIONS TO PROVIDE MAINTENANCE, SUPPORT, UPDATES, ENHANCEMENTS, OR MODIFICATIONS.

In [1]:
import pandas as pd
import gzip
from collections import OrderedDict, defaultdict
import ipaddress
import math
import radix
import re
import numpy as np
import matplotlib.pyplot as plt
import bz2
from fuzzywuzzy import fuzz
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

C:\Users\hamme\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
pd.set_option('display.max_columns', 500)

In [3]:
split = re.compile(r'(.+?):\s*(.*)')

class Entry(OrderedDict):
    def __repr__(self):
        output = []
        for key, value in self.items():
            output.append('{}:\t{}'.format(key, value))
        return '\n'.join(output)
    
    @property
    def date(self):
        # self refers to the object, which in this case is a subclass of dict
        # the .get method for dict objects retrieves the value if the key exists,
        # otherwise it returns the default value
        changed = self.get('changed', None)
        if changed is not None:
            try:
                date = changed.split()[1]
            except IndexError:
                return '17000101'
        else:
            try:
                date = self['last-modified'].replace('-', '')
            except KeyError:
                return '16000101'
        return date
    
    
def parse_whois(filename):
    with gzip.open(filename, 'rt', encoding='latin-1') as f:
        items = []  # list to hold whois items
        item = Entry()  # an object to hold an individual entry

        # Iterate over the lines in the whois file
        for line in f:
            ol = line  # original version of the line
            line = line.strip()  # get rid of whitespace at beginning and end of the line

            # If the original line is not just a newline character,
            # and the line is only whitespace or starts with a comment character,
            # skip the line
            if ol != '\n' and (not line or line[0] == '#'):
                continue

            # If the line is not just whitespace
            if line:
                
                # if original line start with whitespace, append it to previous value
                if re.match(r'^[^a-zA-Z0-9]', ol):
                    try:
                        item[k] += '\n' + line
                        continue
                    except:
                        # print(item)
                        continue

                
                # See if the line matches the regex
                m = split.match(line)
                # If it does
                if m:
                    # There are 2 possible matching groups, so assign them to k and v
                    k, v = m.groups()
                    
                    # If the key is already in the item, concatenate the value to the existing value
                    if k in item and k != 'origin':
                        try:
                            item[k] += '\n' + v
                        except:
                            pass
                            # print(item)
                    # When the key does not yet exist in the item, add the key and value
                    else:
                        try:
                            item[k] = v
                        except:
                            pass
                            # print(item)
                # If it does not match
                else:
                    # Add the value to the previous key in the item
                    # This is a value with newline breaks
                    try:
                        item[k] += '\n' + line
                    except:
                        item[k] = line

            # If the line is just a newline break, finish the current item, and start a new one
            else:
                if item:
                    if set(['organisation', 'inetnum', 'aut-num']) & set(item.keys()):
                        items.append(item)  # Add item to list of items
                    item = Entry()  # Start new item
    return items

In [4]:
def build_as_rel():
    path = "../20260401.as-rel.txt.bz2" # Path to your AS Relationship dataset
    as_rel = defaultdict(lambda: defaultdict(set))
    with bz2.open(path, 'rt') as file:
        for line in file:
            if line.startswith('#'):
                continue
            data = line.strip().split('|')
            AS1 = 'AS'+data[0]
            AS2 = 'AS'+data[1]
            rel = data[2]
            if rel == '0':
                as_rel[AS1]['peer'].add(AS2)
                as_rel[AS2]['peer'].add(AS1)
            elif rel == '1':
                as_rel[AS1]['provider'].add(AS2)
                as_rel[AS2]['customer'].add(AS1)
            elif rel == '-1':
                as_rel[AS1]['customer'].add(AS2)
                as_rel[AS2]['provider'].add(AS1)
    return as_rel


def check_asrel(left, right): # left has to be customer of right
    try:
        for i in left:
            for j in right:
                if i in as_rel[j]['customer']:
                    return True
        return False
    except:
        return False

def build_as2org():
    path = "../20260401.as-org2info.txt.gz"# Path to your AS Organizations dataset
    as2org = gzip.open(path)
    mapping = {}
    companyname = {}
    for line in as2org:
        l = line.decode().strip('\n').split('|')
        asn = None
        if l[0].isdigit():
            asn = 'AS'+str(l[0])
            mapping[asn] = l[3]
        elif len(l) == 5:
            companyname[l[0]] = l[2]
    for i in mapping:
        orgname = companyname[mapping[i]]
        mapping[i] = orgname
    return mapping

def check_as2org(left, right):
    try:
        for i in left:
            for j in right:
                if as2org[i] == as2org[j]:
                    return True
        return False
    except:
        return False

as_rel = build_as_rel()
as2org = build_as2org()

def to_org(asn):
    try:
        return as2org[asn]
    except:
        return None

# Load RIPE DB

In [5]:
ripe = parse_whois("../ripe.db.gz") # Path to your RIPE WHOIS database file
len(ripe)

4283053

In [6]:
ripeaddr = pd.DataFrame([i for i in ripe if 'inetnum' in i])

In [7]:
ripeorg = pd.DataFrame([i for i in ripe if 'organisation' in i])

In [8]:
ripeorg['organisation'] = ripeorg['organisation'].apply(lambda x: x.upper())

In [9]:
ripeas = pd.DataFrame([i for i in ripe if 'aut-num' in i])

In [10]:
print(len(ripeaddr), len(ripeorg), len(ripeas))

4118904 124942 39207


In [11]:
ripeaddr

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size
0,213.159.160.0 - 213.159.191.255,SE-ERICSSON-20010504,DK,ORG-EA44-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nERICSSON-MNT,1970-01-01T00:00:00Z,2022-08-04T09:06:42Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,194.219.52.224 - 194.219.52.239,FORTHNET-NOC-KLI,GR,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,FORTHNETGR-MNT,1970-01-01T00:00:00Z,2018-01-08T15:56:56Z,RIPE,****************************\n* THIS OBJECT IS...,FORTHnet NOC\nKallithea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,217.141.234.232 - 217.141.234.239,INNOTEC,IT,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,INTERB-MNT,1970-01-01T00:00:00Z,2012-06-27T12:05:03Z,RIPE,****************************\n* THIS OBJECT IS...,INNOTEC,network@cgi.interbusiness.it,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,62.27.21.0 - 62.27.21.255,TIS-D402292-NET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS12312-MNT,1970-01-01T00:00:00Z,2009-10-15T08:41:51Z,RIPE,****************************\n* THIS OBJECT IS...,Holger Hartstock\nPostfach 501423\n50974 Koeln,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,212.82.218.48 - 212.82.218.63,HEALTH-GTUA,UA,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,GTUA-WO-MNT,1970-01-01T00:00:00Z,2007-10-01T13:57:33Z,RIPE,****************************\n* THIS OBJECT IS...,"Clinical diagnostic centre ""Health of the elde...",security@svitonline.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4118899,37.206.128.216 - 37.206.128.223,BANCADICREDITOCOOPERATIVODIANAGNI,IT,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,INTERB-MNT,2026-04-03T20:45:36Z,2026-04-03T20:45:36Z,RIPE,****************************\n* THIS OBJECT IS...,BANCA DI CREDITO COOPERATIVO DI ANAGNI,datacomnet@telecomitalia.it,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4118900,195.96.129.128 - 195.96.129.255,InterLIR-Marketplace,NL,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,interlir-mnt,2026-04-03T21:00:18Z,2026-04-03T21:01:07Z,RIPE,****************************\n* THIS OBJECT IS...,InterLIR-Marketplace,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACRO45564-RIPE,https://interlirgeo.github.io/geofeed/195-96-1...,NaN
4118901,195.96.129.0 - 195.96.129.127,InterLIR-Marketplace,NL,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,interlir-mnt,2026-04-03T21:00:20Z,2026-04-03T21:01:09Z,RIPE,****************************\n* THIS OBJECT IS...,InterLIR-Marketplace,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACRO45564-RIPE,https://interlirgeo.github.io/geofeed/195-96-1...,NaN
4118902,51.89.39.24 - 51.89.39.27,OVH_490644791,DE,ORG-MS468-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,OVH-MNT,2026-04-03T21:33:50Z,2026-04-03T21:33:50Z,RIPE,****************************\n* THIS OBJECT IS...,Failover Ips,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Prep for inetnum tree construction

In [12]:
def range_to_pfx(inetnum):
    start, end = inetnum.strip().split('-')
    start_ip = ipaddress.IPv4Address(start.strip())
    end_ip = ipaddress.IPv4Address(end.strip())
    if int(end_ip) - int(start_ip) + 1 < 256:
        return None
    cidr_blocks = ipaddress.summarize_address_range(start_ip, end_ip)
    res = []
    for i in cidr_blocks:
        if i.prefixlen > 24:
            return None
        res.append(str(i))
    return res

In [13]:
ripeaddr['prefix'] = ripeaddr['inetnum'].apply(range_to_pfx)

In [14]:
tree_candidate = ripeaddr.dropna(subset=['prefix'])

In [15]:
len(tree_candidate)

480661

In [16]:
len(ripeaddr) - len(tree_candidate)

3638243

In [17]:
tree_candidate['blocks_in_chunk'] = tree_candidate['prefix'].apply(len)

C:\Users\hamme\AppData\Local\Temp\ipykernel_27376\2053131811.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tree_candidate['blocks_in_chunk'] = tree_candidate['prefix'].apply(len)


In [18]:
tree = tree_candidate.explode('prefix')

In [19]:
tree

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk
0,213.159.160.0 - 213.159.191.255,SE-ERICSSON-20010504,DK,ORG-EA44-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nERICSSON-MNT,1970-01-01T00:00:00Z,2022-08-04T09:06:42Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,213.159.160.0/19,1
3,62.27.21.0 - 62.27.21.255,TIS-D402292-NET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS12312-MNT,1970-01-01T00:00:00Z,2009-10-15T08:41:51Z,RIPE,****************************\n* THIS OBJECT IS...,Holger Hartstock\nPostfach 501423\n50974 Koeln,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.27.21.0/24,1
7,195.194.136.0 - 195.194.139.255,NESCOL-NET,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,JANET-HOSTMASTER,2001-10-24T08:17:49Z,2017-07-31T15:03:05Z,RIPE,Previously - Aberdeen College\n***************...,North East Scotland College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.194.136.0/22,1
13,62.67.0.0 - 62.67.255.255,UK-LVLT-20001108,GB,ORG-LC4-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nLEVEL3-MNT,2002-07-04T09:19:27Z,2016-05-18T12:15:05Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,notify@eu.level3.net,LEVEL3-MNT,LEVEL3-MNT,LEVEL3-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.67.0.0/16,1
15,212.11.160.0 - 212.11.191.255,SA-SPSNET-990409,SA,ORG-SA53-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nwna-mnt,2001-12-17T10:28:26Z,2016-09-15T15:47:25Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,admin@sps.net.sa,SM1805-RIPE-MNT\nwna-mnt,SM1805-RIPE-MNT,SM1805-RIPE-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,212.11.160.0/19,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4118885,193.193.122.0 - 193.193.122.255,VMCBBUK,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS5089-MNT,2026-04-03T19:43:35Z,2026-04-03T19:43:35Z,RIPE,Virgin Media Consumer Broadband UK\nReport Abu...,BIGGLESWADE,ipam@virginmedia.co.uk\nEmail Abuse & Blacklis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.193.122.0/24,1
4118886,2.27.59.0 - 2.27.59.255,ETERNITY-NETWORK,DE,ORG-EIL20-RIPE,DUMY-RIPE,DUMY-RIPE,SUB-ALLOCATED PA,lir-us-acedatacenter-1-MNT,2026-04-03T19:48:25Z,2026-04-03T19:48:25Z,RIPE,****************************\n* THIS OBJECT IS...,EternityCloud,NaN,ETERNITY-MNT,ETERNITY-MNT,ETERNITY-MNT,NaN,NaN,NaN,NaN,ACRO61247-RIPE,https://geofeed.cc/eUGFTP6ZvlW1v2JSGqVzsBp6JhF...,NaN,2.27.59.0/24,1
4118894,132.243.207.0 - 132.243.207.255,InterLIR-Marketplace,GB,NaN,DUMY-RIPE,DUMY-RIPE,LEGACY,interlir-mnt,2026-04-03T19:59:29Z,2026-04-03T21:20:18Z,RIPE,****************************\n* THIS OBJECT IS...,InterLIR-Marketplace,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACRO45564-RIPE,https://interlirgeo.github.io/geofeed/132-243-...,NaN,132.243.207.0/24,1
4118897,161.104.184.0 - 161.104.187.255,Blacknight_Internet_Solutions,IE,NaN,DUMY-RIPE,DUMY-RIPE,LEGACY,us-fxcapital-1-mnt,2026-04-03T20:24:06Z,2026-04-03T20:27:11Z,RIPE,****************************\n* THIS OBJECT IS...,"Unit 12A Barrowside Business Park, Sleaty Road...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,161.104.184.0/22,1


In [20]:
rtree = radix.Radix()
for i in tree.prefix:
    rtree.add(i)

In [21]:
tree.prefix.count()

512054

In [22]:
tree.prefix.nunique()

509387

In [23]:
tree.status.value_counts()

status
ASSIGNED PA              359449
ALLOCATED PA              70811
LEGACY                    21345
ASSIGNED PI               19912
ALLOCATED UNSPECIFIED     15979
SUB-ALLOCATED PA          13859
LIR-PARTITIONED PA         8029
ALLOCATED-ASSIGNED PA      2002
AGGREGATED-BY-LIR           618
ASSIGNED ANYCAST             50
Name: count, dtype: int64

# match allocated PA to ASN and orgs

In [24]:
allocated = tree[tree.status == 'ALLOCATED PA']

In [25]:
allocated['org'] = allocated['org'].apply(lambda x: x.upper())

C:\Users\hamme\AppData\Local\Temp\ipykernel_27376\949084044.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  allocated['org'] = allocated['org'].apply(lambda x: x.upper())


In [26]:
allocated = allocated.merge(ripeorg[['organisation', 'org-name']], left_on='org', right_on='organisation', how='left')

In [27]:
ripeasgroups1 = ripeas.groupby('org').agg({'aut-num':list}).reset_index()

In [28]:
ripeasgroups2 = ripeas.groupby('sponsoring-org').agg({'aut-num':list}).reset_index().rename(columns={'sponsoring-org':'org'})

In [29]:
ripeasgroups = ripeasgroups1.merge(ripeasgroups2, on='org', how='outer')

In [30]:
def combine_autnums(x):
    if isinstance(x['aut-num_x'], float):
        return x['aut-num_y']
    elif isinstance(x['aut-num_y'], float):
        return x['aut-num_x']
    else:
        return list(set(x['aut-num_x'] + x['aut-num_y']))

In [31]:
ripeasgroups['aut-num'] = ripeasgroups.apply(combine_autnums, axis=1)

In [32]:
allocated = allocated.merge(ripeasgroups[['org', 'aut-num']], how='left')

In [33]:
allocated

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,organisation,org-name,aut-num
0,213.159.160.0 - 213.159.191.255,SE-ERICSSON-20010504,DK,ORG-EA44-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nERICSSON-MNT,1970-01-01T00:00:00Z,2022-08-04T09:06:42Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,213.159.160.0/19,1,ORG-EA44-RIPE,Telefonaktiebolaget L M Ericsson,"[AS8479, AS30921, AS35605, AS200832, AS204015,..."
1,62.67.0.0 - 62.67.255.255,UK-LVLT-20001108,GB,ORG-LC4-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nLEVEL3-MNT,2002-07-04T09:19:27Z,2016-05-18T12:15:05Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,notify@eu.level3.net,LEVEL3-MNT,LEVEL3-MNT,LEVEL3-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.67.0.0/16,1,ORG-LC4-RIPE,Lumen Technologies UK Limited,"[AS9057, AS204933, AS199190, AS39628]"
2,212.11.160.0 - 212.11.191.255,SA-SPSNET-990409,SA,ORG-SA53-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nwna-mnt,2001-12-17T10:28:26Z,2016-09-15T15:47:25Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,admin@sps.net.sa,SM1805-RIPE-MNT\nwna-mnt,SM1805-RIPE-MNT,SM1805-RIPE-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,212.11.160.0/19,1,ORG-SA53-RIPE,Gulf Computer Services Co Ldt,[AS42428]
3,195.22.192.0 - 195.22.223.255,IT-SEABONE-970416,IT,ORG-TIIO1-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nAS6762-MNT,2002-06-28T09:03:36Z,2016-08-09T14:36:45Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,tech@seabone.net,AS6762-MNT,NaN,AS6762-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.22.192.0/19,1,ORG-TIIO1-RIPE,TELECOM ITALIA SPARKLE S.p.A.,[AS6762]
4,212.23.224.0 - 212.23.255.255,UK-COLT-19991018,CH,ORG-CI9-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nCOLT-UK,2001-12-17T10:28:13Z,2016-09-15T15:47:27Z,RIPE,Abuse: abuse@colt.net\nAbuse: abuse@colt.net\n...,NaN,NaN,COLT-CH-MNT\nCOLT-UK,NaN,COLT-CH-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,212.23.224.0/19,1,ORG-CI9-RIPE,COLT Technology Services Group Limited,"[AS205730, AS21137, AS47700, AS35724, AS51222,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70806,213.193.234.0 - 213.193.247.255,NL-TRUESERVER-20020628,NL,ORG-TB3-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nTRUESERVER-MNT,2026-04-02T11:11:41Z,2026-04-02T11:11:41Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,sabrina@true.nl,TRUESERVER-MNT,NaN,TRUESERVER-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,213.193.236.0/22,3,ORG-TB3-RIPE,TrueFullstaq B.V.,[AS15703]
70807,213.193.234.0 - 213.193.247.255,NL-TRUESERVER-20020628,NL,ORG-TB3-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,RIPE-NCC-HM-MNT\nTRUESERVER-MNT,2026-04-02T11:11:41Z,2026-04-02T11:11:41Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,sabrina@true.nl,TRUESERVER-MNT,NaN,TRUESERVER-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,213.193.240.0/21,3,ORG-TB3-RIPE,TrueFullstaq B.V.,[AS15703]
70808,153.80.249.0 - 153.80.249.255,RU-NOCRU-19910923,CH,ORG-LT104-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,INETTECH-MNT\nRIPE-NCC-HM-MNT,2026-04-02T11:29:55Z,2026-04-02T16:30:54Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://geofeeds.packetvis.com/8f3fc_153.csv,NaN,153.80.249.0/24,1,ORG-LT104-RIPE,"LLC ""Internet Tehnologii""","[AS28736, AS41748, AS43993, AS49427, AS49530, ..."
70809,85.137.184.0 - 85.137.191.255,AT-FEISTRITZWERKE-STEWEAG-20041103,AT,ORG-FG95-RIPE,DUMY-RIPE,DUMY-RIPE,ALLOCATED PA,at-feistritzwerke-steweag-1-mnt\nRIPE-NCC-HM-MNT,2026-04-02T11:59:56Z,2026-04-02T11:59:56Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.137.184.0/21,1,ORG-FG95-RIPE,Feistritzwerke-STEWEAG GmbH,[AS204410]


# find leaf nodes, suballocated PA or assigned PA

In [34]:
def is_leaf(prefix):
    res = rtree.search_covered(prefix)
    if len(res) == 1:
        return True
    return False

In [35]:
tree['leaf'] = tree['prefix'].apply(is_leaf)

In [36]:
tree['leaf'].value_counts()

leaf
True     466440
False     45614
Name: count, dtype: int64

In [37]:
leafs = tree[(tree.leaf == True) & tree.status.isin(['SUB-ALLOCATED PA', 'ASSIGNED PA'])]

In [38]:
len(leafs)
print(tree["prefix"])

0          213.159.160.0/19
3             62.27.21.0/24
7          195.194.136.0/22
13             62.67.0.0/16
15          212.11.160.0/19
                 ...       
4118885    193.193.122.0/24
4118886        2.27.59.0/24
4118894    132.243.207.0/24
4118897    161.104.184.0/22
4118898    161.104.188.0/22
Name: prefix, Length: 512054, dtype: object


# find root of leaf

In [39]:
def find_root(prefix):
    curr = rtree.search_exact(prefix)
    while curr.parent and curr.parent.prefix != '0.0.0.0/0':
        curr = curr.parent
    return curr.prefix

In [40]:
leafs['root'] = leafs['prefix'].apply(find_root)

C:\Users\hamme\AppData\Local\Temp\ipykernel_27376\3087496323.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leafs['root'] = leafs['prefix'].apply(find_root)


In [41]:
allocated_descr = dict(zip(allocated['prefix'], allocated['descr']))

In [42]:
leafs

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root
3,62.27.21.0 - 62.27.21.255,TIS-D402292-NET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS12312-MNT,1970-01-01T00:00:00Z,2009-10-15T08:41:51Z,RIPE,****************************\n* THIS OBJECT IS...,Holger Hartstock\nPostfach 501423\n50974 Koeln,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.27.21.0/24,1,True,62.26.0.0/15
7,195.194.136.0 - 195.194.139.255,NESCOL-NET,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,JANET-HOSTMASTER,2001-10-24T08:17:49Z,2017-07-31T15:03:05Z,RIPE,Previously - Aberdeen College\n***************...,North East Scotland College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.194.136.0/22,1,True,195.194.0.0/15
50,193.170.79.0 - 193.170.79.255,TUNET-S13,AT,ORG-TUW1-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,ACONET-LIR-MNT,1970-01-01T00:00:00Z,2020-02-05T10:26:27Z,RIPE,for trouble-reports contact <trouble@noc.tuwie...,Technische Universitaet Wien\nVSC Network,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.170.79.0/24,1,True,193.170.0.0/15
51,212.201.51.0 - 212.201.63.255,SLUBNET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,DFN-LIR-MNT,1970-01-01T00:00:00Z,2022-01-21T16:43:08Z,RIPE,****************************\n* THIS OBJECT IS...,Saechsische Landesbibliothek -\nStaats- und Un...,NaN,NaN,NaN,NaN,IRT-DFN-CERT,NaN,NaN,NaN,NaN,NaN,NaN,212.201.51.0/24,3,True,212.201.0.0/16
51,212.201.51.0 - 212.201.63.255,SLUBNET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,DFN-LIR-MNT,1970-01-01T00:00:00Z,2022-01-21T16:43:08Z,RIPE,****************************\n* THIS OBJECT IS...,Saechsische Landesbibliothek -\nStaats- und Un...,NaN,NaN,NaN,NaN,IRT-DFN-CERT,NaN,NaN,NaN,NaN,NaN,NaN,212.201.52.0/22,3,True,212.201.0.0/16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4118848,31.57.167.0 - 31.57.167.255,NET-31-57-167-0-24,BG,ORG-IL896-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,netutils-mnt,2026-04-03T16:04:54Z,2026-04-03T16:04:54Z,RIPE,End User Organization\n***********************...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IL2923-RIPE,https://geofeed.ipxo.com/geofeed.txt,NaN,31.57.167.0/24,1,True,31.56.0.0/14
4118858,78.17.22.0 - 78.17.22.255,Amnezia,NL,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,interlir-mnt,2026-04-03T18:12:58Z,2026-04-03T18:12:58Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACRO45564-RIPE,NaN,NaN,78.17.22.0/24,1,True,78.17.0.0/16
4118859,82.22.167.0 - 82.22.167.255,NET-82-22-167-0-24,CA,ORG-VHI4-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,netutils-mnt,2026-04-03T18:38:30Z,2026-04-03T18:38:30Z,RIPE,End User Organization\n***********************...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VHI4-RIPE,https://geofeed.ipxo.com/geofeed.txt,NaN,82.22.167.0/24,1,True,82.22.0.0/15
4118885,193.193.122.0 - 193.193.122.255,VMCBBUK,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS5089-MNT,2026-04-03T19:43:35Z,2026-04-03T19:43:35Z,RIPE,Virgin Media Consumer Broadband UK\nReport Abu...,BIGGLESWADE,ipam@virginmedia.co.uk\nEmail Abuse & Blacklis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.193.122.0/24,1,True,193.193.96.0/19


# find BGP origin of leaf prefix

In [43]:
bgptree = radix.Radix()

# Path to your prefix2as dataset
with gzip.open('../routeviews-rv2-20260415-1200.pfx2as.gz', 'rt') as file:
    for line in file:
        data = line.strip().split()
        pfx = data[0]
        asns = []
        for i in data[1].split('_'):
            for j in i.split(','):
                asns.append('AS'+j)
        node = bgptree.add(pfx)
        node.data['asn'] = asns

In [44]:
def find_exact_origin(prefix):
    rnode = bgptree.search_exact(prefix)
    if rnode:
        return rnode.data['asn']
    return None

In [100]:
print(leafs.head())
leafs['exact_origin'] = leafs['prefix'].apply(find_exact_origin)
print(leafs.head())

                            inetnum          netname country            org  \
3         62.27.21.0 - 62.27.21.255  TIS-D402292-NET      DE            NaN   
7   195.194.136.0 - 195.194.139.255       NESCOL-NET      GB            NaN   
50    193.170.79.0 - 193.170.79.255        TUNET-S13      AT  ORG-TUW1-RIPE   
51    212.201.51.0 - 212.201.63.255          SLUBNET      DE            NaN   
51    212.201.51.0 - 212.201.63.255          SLUBNET      DE            NaN   

      admin-c     tech-c       status            mnt-by               created  \
3   DUMY-RIPE  DUMY-RIPE  ASSIGNED PA       AS12312-MNT  1970-01-01T00:00:00Z   
7   DUMY-RIPE  DUMY-RIPE  ASSIGNED PA  JANET-HOSTMASTER  2001-10-24T08:17:49Z   
50  DUMY-RIPE  DUMY-RIPE  ASSIGNED PA    ACONET-LIR-MNT  1970-01-01T00:00:00Z   
51  DUMY-RIPE  DUMY-RIPE  ASSIGNED PA       DFN-LIR-MNT  1970-01-01T00:00:00Z   
51  DUMY-RIPE  DUMY-RIPE  ASSIGNED PA       DFN-LIR-MNT  1970-01-01T00:00:00Z   

           last-modified source  \
3  

C:\Users\hamme\AppData\Local\Temp\ipykernel_27376\2634652666.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leafs['exact_origin'] = leafs['prefix'].apply(find_exact_origin)


In [46]:
print(len(leafs[~pd.isnull(leafs.exact_origin)]), len(leafs))

0 370646


In [47]:
leafs['root_origin'] = leafs['root'].apply(find_exact_origin)

C:\Users\hamme\AppData\Local\Temp\ipykernel_27376\808792906.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leafs['root_origin'] = leafs['root'].apply(find_exact_origin)


In [48]:
leafs

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root,exact_origin,root_origin
3,62.27.21.0 - 62.27.21.255,TIS-D402292-NET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS12312-MNT,1970-01-01T00:00:00Z,2009-10-15T08:41:51Z,RIPE,****************************\n* THIS OBJECT IS...,Holger Hartstock\nPostfach 501423\n50974 Koeln,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.27.21.0/24,1,True,62.26.0.0/15,None,None
7,195.194.136.0 - 195.194.139.255,NESCOL-NET,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,JANET-HOSTMASTER,2001-10-24T08:17:49Z,2017-07-31T15:03:05Z,RIPE,Previously - Aberdeen College\n***************...,North East Scotland College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.194.136.0/22,1,True,195.194.0.0/15,None,None
50,193.170.79.0 - 193.170.79.255,TUNET-S13,AT,ORG-TUW1-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,ACONET-LIR-MNT,1970-01-01T00:00:00Z,2020-02-05T10:26:27Z,RIPE,for trouble-reports contact <trouble@noc.tuwie...,Technische Universitaet Wien\nVSC Network,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.170.79.0/24,1,True,193.170.0.0/15,None,None
51,212.201.51.0 - 212.201.63.255,SLUBNET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,DFN-LIR-MNT,1970-01-01T00:00:00Z,2022-01-21T16:43:08Z,RIPE,****************************\n* THIS OBJECT IS...,Saechsische Landesbibliothek -\nStaats- und Un...,NaN,NaN,NaN,NaN,IRT-DFN-CERT,NaN,NaN,NaN,NaN,NaN,NaN,212.201.51.0/24,3,True,212.201.0.0/16,None,None
51,212.201.51.0 - 212.201.63.255,SLUBNET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,DFN-LIR-MNT,1970-01-01T00:00:00Z,2022-01-21T16:43:08Z,RIPE,****************************\n* THIS OBJECT IS...,Saechsische Landesbibliothek -\nStaats- und Un...,NaN,NaN,NaN,NaN,IRT-DFN-CERT,NaN,NaN,NaN,NaN,NaN,NaN,212.201.52.0/22,3,True,212.201.0.0/16,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4118848,31.57.167.0 - 31.57.167.255,NET-31-57-167-0-24,BG,ORG-IL896-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,netutils-mnt,2026-04-03T16:04:54Z,2026-04-03T16:04:54Z,RIPE,End User Organization\n***********************...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IL2923-RIPE,https://geofeed.ipxo.com/geofeed.txt,NaN,31.57.167.0/24,1,True,31.56.0.0/14,None,None
4118858,78.17.22.0 - 78.17.22.255,Amnezia,NL,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,interlir-mnt,2026-04-03T18:12:58Z,2026-04-03T18:12:58Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACRO45564-RIPE,NaN,NaN,78.17.22.0/24,1,True,78.17.0.0/16,None,None
4118859,82.22.167.0 - 82.22.167.255,NET-82-22-167-0-24,CA,ORG-VHI4-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,netutils-mnt,2026-04-03T18:38:30Z,2026-04-03T18:38:30Z,RIPE,End User Organization\n***********************...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VHI4-RIPE,https://geofeed.ipxo.com/geofeed.txt,NaN,82.22.167.0/24,1,True,82.22.0.0/15,None,None
4118885,193.193.122.0 - 193.193.122.255,VMCBBUK,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS5089-MNT,2026-04-03T19:43:35Z,2026-04-03T19:43:35Z,RIPE,Virgin Media Consumer Broadband UK\nReport Abu...,BIGGLESWADE,ipam@virginmedia.co.uk\nEmail Abuse & Blacklis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.193.122.0/24,1,True,193.193.96.0/19,None,None


# Group 3: Child in BGP, parent not in BGP

In [49]:
c1 = leafs[(~pd.isnull(leafs.exact_origin)) & (pd.isnull(leafs.root_origin))]

In [50]:
c1

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root,exact_origin,root_origin


In [51]:
c1 = c1.merge(allocated[['prefix', 'org-name', 'aut-num']].rename(columns={'prefix': 'root'}))

## check if parent assigned ASN is related to child BGP origin AS

In [52]:
c1['sibling'] = c1.apply(lambda x: check_as2org(x['exact_origin'], x['aut-num']), axis=1)

In [53]:
c1['cp'] = c1.apply(lambda x: check_asrel(x['exact_origin'], x['aut-num']), axis=1)

In [54]:
c1infer = c1[(c1.sibling == False) & (c1.cp == False)]

In [55]:
c1infer

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root,exact_origin,root_origin,org-name,aut-num,sibling,cp


# Group 4: Both in BGP, unrelated

In [56]:
c2 = leafs[(~pd.isnull(leafs.exact_origin)) & (~pd.isnull(leafs.root_origin))]
c2 = c2.merge(allocated[['prefix', 'org-name', 'aut-num']].rename(columns={'prefix': 'root'}))

In [57]:
c2['aut_sibling'] = c2.apply(lambda x: check_as2org(x['exact_origin'], x['aut-num']), axis=1)
c2['aut_cp'] = c2.apply(lambda x: check_asrel(x['exact_origin'], x['aut-num']), axis=1)

In [58]:
c2['bgp_sibling'] = c2.apply(lambda x: check_as2org(x['exact_origin'], x['root_origin']), axis=1)
c2['bgp_cp'] = c2.apply(lambda x: check_asrel(x['exact_origin'], x['root_origin']), axis=1)

In [59]:
c2infer = c2[(c2.aut_sibling == False) & (c2.aut_cp == False) & (c2.bgp_sibling == False) & (c2.bgp_cp == False)]

In [60]:
c2infer

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root,exact_origin,root_origin,org-name,aut-num,aut_sibling,aut_cp,bgp_sibling,bgp_cp


# Group 1: Neither in BGP

In [61]:
c3 = leafs[(pd.isnull(leafs.exact_origin)) & (pd.isnull(leafs.root_origin))]

In [62]:
c3

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root,exact_origin,root_origin
3,62.27.21.0 - 62.27.21.255,TIS-D402292-NET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS12312-MNT,1970-01-01T00:00:00Z,2009-10-15T08:41:51Z,RIPE,****************************\n* THIS OBJECT IS...,Holger Hartstock\nPostfach 501423\n50974 Koeln,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,62.27.21.0/24,1,True,62.26.0.0/15,None,None
7,195.194.136.0 - 195.194.139.255,NESCOL-NET,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,JANET-HOSTMASTER,2001-10-24T08:17:49Z,2017-07-31T15:03:05Z,RIPE,Previously - Aberdeen College\n***************...,North East Scotland College,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.194.136.0/22,1,True,195.194.0.0/15,None,None
50,193.170.79.0 - 193.170.79.255,TUNET-S13,AT,ORG-TUW1-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,ACONET-LIR-MNT,1970-01-01T00:00:00Z,2020-02-05T10:26:27Z,RIPE,for trouble-reports contact <trouble@noc.tuwie...,Technische Universitaet Wien\nVSC Network,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.170.79.0/24,1,True,193.170.0.0/15,None,None
51,212.201.51.0 - 212.201.63.255,SLUBNET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,DFN-LIR-MNT,1970-01-01T00:00:00Z,2022-01-21T16:43:08Z,RIPE,****************************\n* THIS OBJECT IS...,Saechsische Landesbibliothek -\nStaats- und Un...,NaN,NaN,NaN,NaN,IRT-DFN-CERT,NaN,NaN,NaN,NaN,NaN,NaN,212.201.51.0/24,3,True,212.201.0.0/16,None,None
51,212.201.51.0 - 212.201.63.255,SLUBNET,DE,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,DFN-LIR-MNT,1970-01-01T00:00:00Z,2022-01-21T16:43:08Z,RIPE,****************************\n* THIS OBJECT IS...,Saechsische Landesbibliothek -\nStaats- und Un...,NaN,NaN,NaN,NaN,IRT-DFN-CERT,NaN,NaN,NaN,NaN,NaN,NaN,212.201.52.0/22,3,True,212.201.0.0/16,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4118848,31.57.167.0 - 31.57.167.255,NET-31-57-167-0-24,BG,ORG-IL896-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,netutils-mnt,2026-04-03T16:04:54Z,2026-04-03T16:04:54Z,RIPE,End User Organization\n***********************...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IL2923-RIPE,https://geofeed.ipxo.com/geofeed.txt,NaN,31.57.167.0/24,1,True,31.56.0.0/14,None,None
4118858,78.17.22.0 - 78.17.22.255,Amnezia,NL,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,interlir-mnt,2026-04-03T18:12:58Z,2026-04-03T18:12:58Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACRO45564-RIPE,NaN,NaN,78.17.22.0/24,1,True,78.17.0.0/16,None,None
4118859,82.22.167.0 - 82.22.167.255,NET-82-22-167-0-24,CA,ORG-VHI4-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,netutils-mnt,2026-04-03T18:38:30Z,2026-04-03T18:38:30Z,RIPE,End User Organization\n***********************...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VHI4-RIPE,https://geofeed.ipxo.com/geofeed.txt,NaN,82.22.167.0/24,1,True,82.22.0.0/15,None,None
4118885,193.193.122.0 - 193.193.122.255,VMCBBUK,GB,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS5089-MNT,2026-04-03T19:43:35Z,2026-04-03T19:43:35Z,RIPE,Virgin Media Consumer Broadband UK\nReport Abu...,BIGGLESWADE,ipam@virginmedia.co.uk\nEmail Abuse & Blacklis...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.193.122.0/24,1,True,193.193.96.0/19,None,None


# Group 2: child not in BGP, parent in BGP

In [63]:
c4 = leafs[(pd.isnull(leafs.exact_origin)) & (~pd.isnull(leafs.root_origin))]

In [64]:
c4

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root,exact_origin,root_origin


# All categories: stats describe

In [65]:
print('Total:', len(leafs))
print('Group 1: Unused:', len(c3))
print('Group 2: Aggregated Customer:', len(c4))
print('Group 3: ISP customer', len(c1) - len(c1infer), 'leased prefixes:', len(c1infer))
print('Group 4: Delegated customer', len(c2) - len(c2infer), 'leased prefixes:', len(c2infer))

Total: 370646
Group 1: Unused: 370646
Group 2: Aggregated Customer: 0
Group 3: ISP customer 0 leased prefixes: 0
Group 4: Delegated customer 0 leased prefixes: 0


# Inferred leases

In [66]:
ripe_inferred_lease = pd.concat([c1infer, c2infer])

In [67]:
leased_prefix = set(ripe_inferred_lease['prefix'])

In [68]:
ripe_inferred_lease['country'].value_counts()[:20]

Series([], Name: count, dtype: int64)

In [69]:
ripe_inferred_lease = ripe_inferred_lease.merge(allocated[['prefix', 'country']], left_on=['root'], right_on=['prefix'], suffixes=('', '_root'))

In [70]:
ripe_inferred_lease[ripe_inferred_lease['org-name'] == 'Resilans AB']['country'].nunique()

0

In [71]:
ripe_inferred_lease['out_of_country'] = ripe_inferred_lease['country'] != ripe_inferred_lease['country_root']

In [72]:
ripe_inferred_lease['out_of_country'].value_counts()

Series([], Name: count, dtype: int64)

In [73]:
ripe_inferred_lease[ripe_inferred_lease.out_of_country]['country'].value_counts()[:20]

Series([], Name: count, dtype: int64)

## Top IP Holders

In [74]:
ripe_inferred_lease['org-name'].value_counts()[:10]

Series([], Name: count, dtype: int64)

## Top Originator

In [75]:
originator = pd.DataFrame(ripe_inferred_lease['exact_origin'].explode().value_counts().reset_index())

In [76]:
originator['org'] = originator['exact_origin'].apply(to_org)

In [77]:
originator[:20]

,exact_origin,count,org


## Top maintainer

In [78]:
mnts = ripe_inferred_lease['mnt-by'].apply(lambda x: x.split('\n')).explode().value_counts().reset_index()

In [79]:
orgdict = defaultdict(set)
for a,b,c in zip(ripeorg['org-name'], ripeorg['mnt-ref'], ripeorg['mnt-by']):
    for i in b.split('\n'):
        if i != 'RIPE-NCC-RIS-MNT' and i != 'RIPE-NCC-HM-MNT':
            orgdict[i].add(a)
    for i in c.split('\n'):
        if i != 'RIPE-NCC-RIS-MNT' and i != 'RIPE-NCC-HM-MNT':
            orgdict[i].add(a)

In [80]:
mnts['org'] = mnts['mnt-by'].apply(lambda x: orgdict[x])

In [81]:
mnts[:20]

,mnt-by,count,org


# Reference Dataset: RIPE Broker Leased Prefixes

In [82]:
brokers = []
with open('../recognized_brokers_ripe.txt', 'rt') as f:
    for line in f:
        brokers.append(line.strip().upper().replace('\"', ''))

In [83]:
ripeorg['org-name'] = ripeorg['org-name'].apply(lambda x: x.upper().replace('\"', ''))

In [84]:
ripeorg[ripeorg['org-name'].isin(brokers)]['org-name'].nunique()

40

In [85]:
org_to_check = set(brokers) - set(ripeorg[ripeorg['org-name'].isin(brokers)]['org-name'].unique())

In [86]:
orgnames = ['ADDREX, INC.',
           'CLAUS PLACHETKA TRADING AS AIPI E. K.',
           'ALFA TELECOM S.R.O.',
           'ALFA TELECOM CJSC',
           'ALFA TELECOM LTD.',
           'ALPHA INFOLAB, INC.',
           'ASR-E DANESH AFZAR COMPANY (PRIVATE J.S.)',
           'AZERONLINE LTD JOINT ENTERPRISE',
           'BLUE NETWORKS TECHNOLOGIES SARL',
           'BRANDER GROUP INC.',
           'E-MONEY NET DEVELOPERS 24 COMPANY PRIVATE JOINT STOCK',
           'FIBERCLI PROYECTOS E INNOVACION S.L.',
           'IP COM LTD',
           'GCX US LLC',
           'I7 LLC',
           'INANA GROUP LLC',
           'INTELLIGENT ADVANCED SOLUTIONS S.R.O',
           'IPWAY LLC',
           'NATIONWIDE COMPUTER SYSTEMS, INC. TRADING AS IPTRADING.COM',
           'IT-SERVICE LLC',
           'KONNGRUENT MANAGEMENT S.R.L.',
           'VLADIMIR KORABLEV',
           'LEADERTELECOM LTD.',
           'LEADERTELECOM B.V.',
           'LLC LIR UKRAINE',
           'NETERRA LTD.',
           'NITRONET SP. Z O.O.',
           'PARSUN NETWORK SOLUTIONS PTY LTD',
           'PREFIX BROKER BV',
           'RIOTCLOUDS, INC.',
           'SEVEN NETWORKING UK LTD',
           'THERECOM LTD',
           'VOLDETA UAB',
           'ULF KIEBER',
           'WINDMILL TELECOM OU',
           'XTOM OU',
             'XTOM GMBH',
             'XTOM GLOBAL TELECOM INC.',
         'XTOM PTY LTD',
         'XTOM PTY LTD',
         'XTOM GMBH',
         'XTOM JAPAN CO., LTD.']

In [87]:
brokers += orgnames

In [88]:
ripeorg[ripeorg['org-name'].isin(brokers)]['org-name'].nunique()

73

In [89]:
brokermnt = set()
for i in ripeorg[ripeorg['org-name'].isin(brokers)]['mnt-by']:
    for j in i.strip().split('\n'):
        if j.strip() != 'RIPE-NCC-HM-MNT':
            brokermnt.add(j.strip())
for i in ripeorg[ripeorg['org-name'].isin(brokers)]['mnt-ref']:
    for j in i.strip().split('\n'):
        if j.strip() != 'RIPE-NCC-HM-MNT':
            brokermnt.add(j.strip())

In [90]:
len(brokermnt)

85

In [91]:
brokerstr = '|'.join(brokermnt)

In [92]:
tree[tree['mnt-by'].str.contains(brokerstr) & (tree.status == 'LEGACY')]

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf
2165,192.93.225.0 - 192.93.225.255,SETRA-NET2,FR,NaN,DUMY-RIPE,DUMY-RIPE,LEGACY,SETRA2-MNT\nCELESTE-MNT,2002-07-03T08:59:29Z,2024-04-25T08:14:49Z,RIPE,****************************\n* THIS OBJECT IS...,CEREMA\n110 Rue de Paris - BP 214\n77487 PROVI...,NaN,NaN,NaN,COMPLETEL-MNT\nMNT-LYONIX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.93.225.0/24,1,True
2166,192.93.226.0 - 192.93.226.255,SETRA-NET3,FR,NaN,DUMY-RIPE,DUMY-RIPE,LEGACY,SETRA2-MNT\nLDCOM-MNT\nCEREMA-MNT,2002-07-03T08:49:17Z,2024-04-25T08:14:49Z,RIPE,****************************\n* THIS OBJECT IS...,CEREMA\n110 Rue de Paris - BP 214\n77487 PROVI...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.93.226.0/24,1,True
2880,192.91.186.0 - 192.91.186.255,DEMOS-SEC,RU,ORG-RCS23-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,RIPE-NCC-LEGACY-MNT\ncz-relcom-1-mnt\nAS2578-MNT,1970-01-01T00:00:00Z,2026-02-15T11:08:53Z,RIPE,****************************\n* THIS OBJECT IS...,-----BEGIN TOKEN-----29e1914fcfc26b84b81143e10...,NaN,interlir-mnt,interlir-mnt,interlir-mnt,NaN,NaN,NaN,NaN,AR63624-RIPE,NaN,NaN,192.91.186.0/24,1,True
37334,156.150.0.0 - 156.150.255.255,EU-ORIGIN-156-150-0-0,NL,ORG-OB2-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,RIPE-NCC-LEGACY-MNT\nGIPC-ORIGIN-MNT,1970-01-01T00:00:00Z,2018-04-19T14:29:36Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,Global-IPcoordinator@atos.net,GIPC-ORIGIN-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,156.150.0.0/16,1,True
37335,192.68.230.0 - 192.68.230.255,Atos-MEV,NL,ORG-OB2-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,RIPE-NCC-LEGACY-MNT\nGIPC-ORIGIN-MNT,1970-01-01T00:00:00Z,2017-08-24T08:32:00Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,Global-IPcoordinator@atos.net,NaN,NaN,MNT-VALUESOLUTIONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.68.230.0/24,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4118197,132.243.206.0 - 132.243.206.255,GLOBALTECH-LLC,FI,ORG-NHTL5-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,interlir-mnt,2026-04-03T04:07:03Z,2026-04-03T04:07:03Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NHT-MNT,NHT-MNT,NaN,NaN,NaN,NaN,ACRO45564-RIPE,https://landvps.ru/ls_geofeed.csv,NaN,132.243.206.0/24,1,True
4118521,132.243.210.0 - 132.243.211.255,BAXET-GROUP-INC,US,ORG-BGI3-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,interlir-mnt,2026-04-03T08:57:05Z,2026-04-03T08:57:05Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ACRO45564-RIPE,https://geofeed.info/geofeed-net132.243.210.0-...,NaN,132.243.210.0/23,1,True
4118523,132.243.209.0 - 132.243.209.255,BAXET-GROUP-INC,US,ORG-BGI3-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,interlir-mnt,2026-04-03T09:01:11Z,2026-04-03T09:01:11Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR34875-RIPE,https://geofeed.info/geofeed-net132.243.209.0-...,NaN,132.243.209.0/24,1,True
4118527,132.243.208.0 - 132.243.208.255,BAXET-GROUP-INC,US,ORG-BGI3-RIPE,DUMY-RIPE,DUMY-RIPE,LEGACY,interlir-mnt,2026-04-03T09:04:04Z,2026-04-03T09:04:04Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR34875-RIPE,https://geofeed.info/geofeed-net132.243.208.0-...,NaN,132.243.208.0/24,1,True


In [93]:
def check_mntner(x):
    x = x.strip().split('\n')
    x = set([i.strip() for i in x])
    if len(x.intersection(brokermnt)) > 0:
        return True
    return False

In [94]:
leafs['true_label'] = leafs['mnt-by'].apply(check_mntner)

C:\Users\hamme\AppData\Local\Temp\ipykernel_27376\2368286913.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leafs['true_label'] = leafs['mnt-by'].apply(check_mntner)


In [95]:
tp = leafs[(leafs.true_label == True)]

## Caution: Not all broker prefixes are leased. Some brokers are also ISPs and provide connectivity to customers  
Filter prefixes that were used by ISPs/brokers on their customers.

In [96]:
tp = tp.merge(allocated[['prefix', 'org-name', 'aut-num']].rename(columns={'prefix': 'root'}))

In [97]:
tp['aut_sibling'] = tp.apply(lambda x: check_as2org(x['exact_origin'], x['aut-num']), axis=1)
tp['aut_cp'] = tp.apply(lambda x: check_asrel(x['exact_origin'], x['aut-num']), axis=1)

tp['bgp_sibling'] = tp.apply(lambda x: check_as2org(x['exact_origin'], x['root_origin']), axis=1)
tp['bgp_cp'] = tp.apply(lambda x: check_asrel(x['exact_origin'], x['root_origin']), axis=1)

## Leased broker prefixes

In [98]:
tp[(tp.aut_sibling == False) & (tp.aut_cp == False) & (tp.bgp_sibling == False) & (tp.bgp_cp == False)]

,inetnum,netname,country,org,admin-c,tech-c,status,mnt-by,created,last-modified,source,remarks,descr,notify,mnt-lower,mnt-domains,mnt-routes,mnt-irt,sponsoring-org,geoloc,language,abuse-c,geofeed,assignment-size,prefix,blocks_in_chunk,leaf,root,exact_origin,root_origin,true_label,org-name,aut-num,aut_sibling,aut_cp,bgp_sibling,bgp_cp
0,212.85.128.0 - 212.85.129.255,NEURONNEXION,FR,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,NEURONNEXION-NOC,1970-01-01T00:00:00Z,2022-05-16T08:01:14Z,RIPE,****************************\n* THIS OBJECT IS...,Neuronnexion Infrastructure - Amiens,NaN,NaN,NaN,NaN,NaN,NaN,49.89239502900903 2.2894906997680664,NaN,NaN,NaN,NaN,212.85.128.0/23,1,True,212.85.128.0/20,None,None,True,Neuronnexion SCOP,"[AS31235, AS57284, AS9036, AS50620, AS44842]",False,False,False,False
1,212.85.136.0 - 212.85.136.255,NEURONNEXION,FR,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,NEURONNEXION-NOC,1970-01-01T00:00:00Z,2016-02-23T10:06:38Z,RIPE,****************************\n* THIS OBJECT IS...,Neuronnexion Infrastructure - Courbevoie\nDedi...,noc@nnx.com,NEURONNEXION-NOC,NaN,NaN,NaN,NaN,48.90377176147852 2.258291244506836,NaN,NaN,NaN,NaN,212.85.136.0/24,1,True,212.85.128.0/20,None,None,True,Neuronnexion SCOP,"[AS31235, AS57284, AS9036, AS50620, AS44842]",False,False,False,False
2,62.217.128.0 - 62.217.143.255,AZERONLINE,AZ,NaN,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AZERONLINE-MNT,2001-12-04T19:49:04Z,2025-09-22T07:30:41Z,RIPE,****************************\n* THIS OBJECT IS...,Azeronline Information Services\nISP of Azerba...,ripe_adm@azeronline.net,NaN,NaN,NaN,NaN,NaN,40.4002 49.8096,NaN,NaN,https://webapi.azeronline.com/storage/1980/geo...,NaN,62.217.128.0/20,1,True,62.217.128.0/19,None,None,True,"""AZERONLINE LTD"" JOINT ENTERPRISE",[AS15723],False,False,False,False
3,194.87.101.0 - 194.87.101.255,BNK-MSK-RU-RUVDS,RU,ORG-JME1-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,interlir-mnt,1970-01-01T00:00:00Z,2024-06-22T09:05:33Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,MTF-MNT,MTF-MNT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,194.87.101.0/24,1,True,194.87.0.0/16,None,None,True,Reliable Communications s.r.o.,"[AS21082, AS2118, AS207700]",False,False,False,False
4,193.124.23.0 - 193.124.23.255,R_HOST,RU,ORG-RHL16-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,AS2118-MNT\ninterlir-mnt,1970-01-01T00:00:00Z,2025-08-11T09:16:07Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,msknoc@relcom.net,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,193.124.23.0/24,1,True,193.124.22.0/23,None,None,True,Reliable Communications s.r.o.,"[AS21082, AS2118, AS207700]",False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6040,78.17.60.0 - 78.17.60.255,Snowd-Security-OU,DE,ORG-SSO10-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,interlir-mnt,2026-04-03T10:22:02Z,2026-04-03T10:22:02Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR34875-RIPE,NaN,NaN,78.17.60.0/24,1,True,78.17.0.0/16,None,None,True,RCS Technologies FZE LLC,[AS216070],False,False,False,False
6041,104.239.89.0 - 104.239.89.255,Azari-Labs,GE,ORG-VU28-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,lir-uk-fastplanet-1-MNT\nvoldeta-mnt,2026-04-03T10:56:30Z,2026-04-03T12:11:59Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR67259-RIPE,NaN,NaN,104.239.89.0/24,1,True,104.239.64.0/18,None,None,True,FASTPLANET LTD,NaN,False,False,False,False
6042,104.233.58.0 - 104.233.58.255,Azari-Labs,GE,ORG-VU28-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,lir-uk-fastplanet-1-MNT\nvoldeta-mnt,2026-04-03T10:57:14Z,2026-04-03T12:12:26Z,RIPE,****************************\n* THIS OBJECT IS...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AR67259-RIPE,NaN,NaN,104.233.58.0/24,1,True,104.233.32.0/19,None,None,True,FASTPLANET LTD,NaN,False,False,False,False
6043,104.239.29.0 - 104.239.29.255,Azari-Labs,GE,ORG-VU28-RIPE,DUMY-RIPE,DUMY-RIPE,ASSIGNED PA,lir-uk-fastplanet-1-MNT\nvol